# Validation tests

In [ ]:
#installation packages
!pip install -q cupy-cuda12x
%pip install QuantLib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 100.3 MB/s eta 0:00:00


In [ ]:
#libraries
import QuantLib as ql
import cupy as cp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.stats import norm
import time


/usr/local/lib/python3.13/dist-packages/cupy/_environment.py:677: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy-cuda12x, cupy-cuda13x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''


## SciPy Implementation Regular

This code implements a CPU-based semi-analytical Heston European call-option pricer

Input: Heston model params
Output: Theoretical price of European call option

In [ ]:
def heston_charac_function_cpu(
        u,
        S0,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho
):
    i = 1j
    d = np.sqrt(
        (rho*sigma*i*u-kappa)**2
        + sigma**2 * (i * u + u**2)
    )
    g = (kappa - rho * sigma * i * u - d)/(kappa - rho * sigma * i * u + d)
    C = (r * i * u * T
        + (kappa * theta / sigma**2)
        *((kappa - rho * sigma * i * u - d) * T
            - 2
            * np.log(
                (1-g * np.exp(-d*T))
                / (1-g)
            )
        )
    )
    D = (
        (kappa - rho * sigma * i * u - d)
        / sigma ** 2
        * (
            (1 - np.exp(-d * T))
            / (1 - g * np.exp(-d * T))
        )
    )
    return np.exp(
        C
        + D * v0
        + i * u * np.log(S0)
    )

def heston_probability_cpu(
        j,
        S0,
        K,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho,
        upper_limit=100 #runs convergence tests with 50, 100, 200, etc.
):
    phi_minus_i = heston_charac_function_cpu(
        -1j,
        S0,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho
    )
    def integrand(u):
        phi_u = heston_charac_function_cpu(
            u,
            S0,
            T,
            r,
            v0,
            kappa,
            theta,
            sigma,
            rho
        )
        if j == 1:
            phi_shifted = heston_charac_function_cpu(
                u-1j,
                S0,
                T,
                r,
                v0,
                kappa,
                theta,
                sigma,
                rho
            )
            value = (
                np.exp(-1j * u * np.log(K))
                * phi_shifted
                / (1j * u * phi_minus_i)
            )
        else:
            value = (
                np.exp(-1j * u * np.log(K))
                * phi_u
                /(1j * u)
            )
        return np.real(value)
    integral, error = quad(
        integrand,
        1e-8,
        100,
        limit=500
    )
    return 0.5 + integral/np.pi

def heston_price_scipy_cpu(
        S0,
        K,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho
):
    P1 = heston_probability_cpu(
        1,
        S0,
        K,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho
    )

    P2 = heston_probability_cpu(
        2,
        S0,
        K,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho
    )

    price = (
        S0 * P1 - K * np.exp(-r * T) * P2
    )

    return price

In [ ]:
#testing CPU implementation
S0 = 100
K = 100
T = 1.0
r = 0.03
v0 = 0.04
kappa = 2.0
theta = 0.04
sigma = 0.30
rho = -0.70

price = heston_price_scipy_cpu(
    S0,
    K,
    T,
    r,
    v0,
    kappa,
    theta,
    sigma,
    rho
)

#testing characteristic function
phi0 = heston_charac_function_cpu(
    0.0,
    S0,
    T,
    r,
    v0,
    kappa,
    theta,
    sigma,
    rho
)

#testing p1 and p2 values
p1 = heston_probability_cpu(
    1,
    S0,
    K,
    T,
    r,
    v0,
    kappa,
    theta,
    sigma,
    rho
)
p2 = heston_probability_cpu(
    2,
    S0,
    K,
    T,
    r,
    v0,
    kappa,
    theta,
    sigma,
    rho
)

#testing no arbitrage bounds
lower_bound = max(
    S0 - K * np.exp(-r * T),
    0
)
upper_bound = S0
assert lower_bound <= price <= upper_bound

print("Test Heston Price: ", price)
print("Test Characteristic Function: ", phi0) #should always be 1+0j
print("Test P1: ", p1) #0\leq p1 \leq 1
print("Test P2: ", p2) #0 \leq p2 \leq 1
print("No aribtrage test passed")

Test Heston Price:  9.242521061494699
Test Characteristic Function:  (1+0j)
Test P1:  0.6556511539247819
Test P2:  0.5803787269238658
No aribtrage test passed


## SciPy Implementation GPU

In [ ]:
'''
def heston_charac_function_gpu(
        u,
        S0,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho
):
    i = 1j
    d = cp.sqrt(
        (rho*sigma*i*u-kappa)**2
        + sigma**2 * (i * u + u**2)
    )
    g = (kappa - rho * sigma * i * u - d)/(kappa - rho * sigma * i * u + d)
    C = (r * i * u * T
        + (kappa * theta / sigma **2)
        *(
            (kappa - rho * sigma * i * u - d) * T
            - 2 * cp.log((1-g * cp.exp(-d*T))/ (1-g))))
    D = (
        (kappa - rho * sigma * i * u - d)
        / sigma ** 2
        * (
            (1 - cp.exp(-d * T))
            / (1 - g * cp.exp(-d * T))
        )
    )
    return cp.exp(
        C
        + D * v0
        + i * u * cp.log(S0)
    )

def heston_probability_gpu(
        j,
        S0,
        K,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho,
        N=20_000,
        u_max=100
):
    u = cp.linspace(
        1e-8,
        u_max,
        N,
        dtype=cp.float64
    )

    phi_minus_i = heston_charac_function_gpu(
        cp.asarray(-1j),
        S0,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho
    )
    if j == 1:
      phi_shifted = heston_charac_function_gpu(
          u-1j,
          S0,
          T,
          r,
          v0,
          kappa,
          theta,
          sigma,
          rho
      )
      integrand = cp.real(
          cp.exp(-1j * u * cp.log(K)) * phi_shifted / (1j * u * phi_minus_i)
      )
    else:
      phi_u = heston_charac_function_gpu(
          u,
          S0,
          T,
          r,
          v0,
          kappa,
          theta,
          sigma,
          rho
      )
      integrand = cp.real(
          cp.exp(-1j * u * cp.log(K)) * phi_u / (1j * u)
      )
    integral = cp.trapezoid(integrand, u)
    probability = (0.5 + integral/cp.pi)
    return probability

def heston_price_gpu(
        S0,
        K,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho,
        N=20_000,
        u_max=100
):
    S0 = cp.asarray(S0, dtype=cp.float64),
    K = cp.asarray(K, dtype=cp.float64),
    T = cp.asarray(T, dtype=cp.float64),
    r = cp.asarray(r, dtype=cp.float64),
    v0 = cp.asarray(v0, dtype=cp.float64),
    kappa = cp.asarray(kappa, dtype=cp.float64),
    theta = cp.asarray(theta, dtype=cp.float64),
    sigma = cp.asarray(sigma, dtype=cp.float64),
    rho = cp.asarray(rho, dtype=cp.float64),

    P1 = heston_probability_gpu(
        1,
        S0,
        K,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho,
        N=N,
        u_max=u_max
    )

    P2 = heston_probability_gpu(
        2,
        S0,
        K,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho,
        N=N,
        u_max=u_max
    )

    price = (
        S0 * P1 - K * cp.exp(-r * T) * P2
    )

    return float(cp.asnumpy(price))
'''

'\ndef heston_charac_function_gpu(\n        u,\n        S0,\n        T,\n        r,\n        v0,\n        kappa,\n        theta,\n        sigma,\n        rho\n):\n    i = 1j\n    d = cp.sqrt(\n        (rho*sigma*i*u-kappa)**2\n        + sigma**2 * (i * u + u**2)\n    )\n    g = (kappa - rho * sigma * i * u - d)/(kappa - rho * sigma * i * u + d)\n    C = (r * i * u * T\n        + (kappa * theta / sigma **2)\n        *(\n            (kappa - rho * sigma * i * u - d) * T\n            - 2 * cp.log((1-g * cp.exp(-d*T))/ (1-g))))\n    D = (\n        (kappa - rho * sigma * i * u - d)\n        / sigma ** 2\n        * (\n            (1 - cp.exp(-d * T))\n            / (1 - g * cp.exp(-d * T))\n        )\n    )\n    return cp.exp(\n        C\n        + D * v0\n        + i * u * cp.log(S0)\n    )\n\ndef heston_probability_gpu(\n        j,\n        S0,\n        K,\n        T,\n        r,\n        v0,\n        kappa,\n        theta,\n        sigma,\n        rho,\n        N=20_000,\n        u_max=10

## QuantLib Implementation

In [ ]:
#reusing quantlib function from before
def heston_quantlib_prices(
    S0, #stock price today
    K, #strike price
    T, #time to maturity (in years)
    r, #risk free interest rate
    v0, #variance today
    kappa, #speed of mean reversion
    theta, #long run variance
    sigma, #volatility of variance
    rho, #correlation between stock and variance
    q=0.0 #dividend yield
):
    #date setup
    today = ql.Date.todaysDate()
    ql.Settings.instance().evaluationDate = today

    #conversion of T into maturity date
    maturity_date = today + int(round(T * 365))

    #day count conversion
    day_count = ql.Actual365Fixed()

    #spot price
    spot = ql.QuoteHandle(
        ql.SimpleQuote(S0)
    )

    #risk free rate
    risk_free_curve = ql.YieldTermStructureHandle(
        ql.FlatForward(today,r,day_count)
    )

    dividend_curve = ql.YieldTermStructureHandle(
        ql.FlatForward(today,q,day_count)
    )

    heston_process = ql.HestonProcess(
        risk_free_curve,
        dividend_curve,
        spot,
        v0,
        kappa,
        theta,
        sigma,
        rho
    )

    #heston model
    heston_model = ql.HestonModel(heston_process)

    #analytical heston pricing engine
    engine = ql.AnalyticHestonEngine(heston_model)

    #European call
    payoff = ql.PlainVanillaPayoff(
        ql.Option.Call,
        K
    )

    #
    exercise = ql.EuropeanExercise(
        maturity_date
    )

    option = ql.VanillaOption(
        payoff,
        exercise
    )

    option.setPricingEngine(engine)

    return option.NPV()

## Black Scholes
Implementation for validation testing later.

In [ ]:
def black_scholes_call(
    S0,
    K,
    T,
    r,
    sigma_b_s,
    q=0.0
):
  d1 = (
      np.log(S0/K)
      + (
          r - q + 0.5 * sigma_b_s ** 2
      ) * T
  ) / (sigma_b_s * np.sqrt(T)
  )

  d2 = (
      d1 - sigma_b_s * np.sqrt(T)
  )
  price = (
      S0 * np.exp(-q * T) * norm.cdf(d1)
      - K * np.exp(-r * T) * norm.cdf(d2)
  )
  return price

## Base Parameters

In [ ]:
base_params = {
    "S0":100.0,
    "K":100.0,
    "T":1.0,
    "r":0.03,
    "v0":0.04,
    "kappa":2.0,
    "theta":0.04,
    "sigma":0.30,
    "rho":-0.70
}

## Basic Sanity Tests

In [ ]:
price = heston_price_scipy_cpu(**base_params)
phi0 = heston_charac_function_cpu(
    0.0,
    base_params["S0"],
    base_params["T"],
    base_params["r"],
    base_params["v0"],
    base_params["kappa"],
    base_params["theta"],
    base_params["sigma"],
    base_params["rho"]
)
P1 = heston_probability_cpu(
    1,
    **base_params
)
P2 = heston_probability_cpu(
    2,
    **base_params
)
print("Heston Price:", price),
print("phi(0):", phi0),
print("P1:", P1),
print("P2:", P2)

#characteristic function identity
assert np.isclose(
    phi0,
    1.0+0j,
    atol=1e-10
  )

#probability bounds
assert 0 <= P1 <= 1
assert 0 <= P2 <= 1

upper_bound = (base_params["S0"])

assert(lower_bound <= price <= upper_bound)

print("Basic Sanity Tests Passed")

## Strike Monotonicty Test

In [ ]:
strikes = [
    80,
    90,
    100,
    110,
    120
]

strikes_prices = []

for K in strikes:
  test_params = base_params.copy()
  test_params["K"] = K
  price = heston_price_scipy_cpu(**test_params)
  strikes_prices.append(price)

assert all(
    strikes_prices[i] >= strikes_prices[i+1]
    for i in range(len(strikes_prices)-1)
)

print("Strike Monotonicty Test Passed")

# Comparison Function
Compares both SciPy and QuantLib implementations

In [ ]:
def compare_pricers(params):
    ql_price = heston_quantlib_prices(**params)
    cpu_price = heston_price_scipy_cpu(**params)

    sign_error = (cpu_price - ql_price)
    abs_error = abs(sign_error)
    rel_error = (abs_error / max(abs(ql_price), 1e-12))
    return{
        **params,
        "QuantLib Price": ql_price,
        "CPU Price": cpu_price,
        "Absolute Error": abs_error,
        "Relative Error": rel_error
    }

## Validation Test:
Single Ref Case

In [ ]:
single_result = compare_pricers(base_params)
print("Single Ref Case:")
print("--------------")
print(
    f"QuantLib",
    f"{single_result['QuantLib Price']:.10f}"
)

print(
    f"SciPy",
    f"{single_result['CPU Price']:.10f}"
)

print(
    f"Absolute Error",
    f"{single_result['Absolute Error']:.10f}"
)

Strucutred strike and maturity validation

In [ ]:
def structured_validation():
  strikes = [60,80,100,120,140]
  maturity_days = [30,90,180,365,730]
  results = []
  for days in maturity_days:
    T = days/365.0
    for K in strikes:
      params = base_params.copy()
      params["K"] = float(K)
      params["T"] = T
      comparison = (compare_pricers(params))
      results.append({
          "K":K,
          "Days": days,
          "T": T,
          **comparison
      })
  df = pd.DataFrame(results)
  mae = (df["Absolute Error"].mean())
  rmse = np.sqrt(np.mean(df["Relative Error"] ** 2))
  max_error = (df["Absolute Error"].max())
  mean_signed_error = (df["Relative Error"].mean())

  print(f"Number of cases: {len(df)}")
  print(f"MAE: {mae:.10f}")
  print(f"RMSE: {rmse:.10f}")
  print(f"Max Error: {max_error:.10f}")
  return df

strucuteded_validation = structured_validation()

In [ ]:
strike_plot_results = []
for K in np.arange(60,141,10):
  params = base_params.copy()
  params["K"] = float(K)
  strike_plot_results.append(compare_pricers(params))
df_strikes = pd.DataFrame(strike_plot_results)
plt.figure(figsize=(8,5))
plt.plot(
    df_strikes["K"],
    df_strikes["QuantLib Price"],
    marker="o"
    label="Quantlib"
)

plt.plot(
    df_strikes["K"],
    df_strikes["CPU Price"],
    marker="o",
    label="CPU"
)

plt.xlabel("Strike Price K")
plt.ylabel("European Call Price")
plt.title("Heston Price vs. Strike")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

Validation Test 3: No arbitrage call bounds

In [ ]:
price = heston_price_scipy_cpu(**params)
lower_bound = max(
    params["S0"] - params["K"] * np.exp(-params["r"] * params["T"]),
    0
)
upper_bound = params["S0"]

print(f"Lower bound: {lower_bound:.6f}")
print(f"Heston price: {price: .6f}")
print(f"Upper bound: {upper_bound:.6f}")

assert lower_bound <= price <= upper_bound

print("No-arbitrage bounds passed")

Validation Test
Randomized CPU against QuanLib

In [ ]:
def randomized_validation(n_samples=25, seed=42):
  rng = np.random.default_rng(seed)
  results = []
  for i in range(n_samples):
    S0 = 100.0
    K = rng.uniform(60.0,140.0)
    days = rng.integers(30,731)
    T = days/365.0
    r = rng.uniform(0.0,0.08)
    v0 = rng.uniform(0.01,0.16)
    kappa = rng.uniform(0.5,5.0)
    theta = rng.uniform(0.01,0.16)
    sigma = rng.uniform(0.1,1.0)
    rho = rng.uniform(-0.95, -0.05)
    params = {
        "S0": S0,
        "K": K,
        "T": T,
        "r": r,
        "v0": v,
        "kappa":kappa,
        "theta":theta,
        "sigma":sigma,
        "rho":rho
    }
    comparison = compare_cpu_quantlib(params)
    feller_value =(2 * kappa * theta - sigma**2)
    feller_satisfied = (feller_value >= 0)
    results.append({
        **params,
        "Days": days,
        **comparison,
        "Feller": feller_satisfied
    })
  df = pd.DataFrame(results)
  print("Randomized CPU vs. QuantLib validation")
  mae = (df["Absolute Error"].mean())
  rmse = np.sqrt(np.mean(df["Signed Error"] ** 2))
  max_error = (df["Absolute Error"].max())
  mean_signed_error = (df["Signed Error"].mean())

  print(f"Number of cases: {len(df)}")
  print(f"MAE: {mae:.10f}")
  print(f"RMSE: {rmse:.10f}")
  print(f"Max Error: {max_error:.10f}")
  return df


Validation Test: Comparison of Quantlib and CPU against Black-Scholes